In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
cd /content/drive/MyDrive/KYUTECH/Lab research/Research/Anomaly Detection ADCS/Coding/Noisy dataset analysis detection/

/content/drive/MyDrive/KYUTECH/Lab research/Research/Anomaly Detection ADCS/Coding/Noisy dataset analysis detection


In [ ]:
# cd /content/drive/MyDrive/KYUTECH/Lab research/Research/Anomaly Detection ADCS/Coding/Notebooks_detection/

/content/drive/MyDrive/KYUTECH/Lab research/Research/Anomaly Detection ADCS/Coding/Notebooks_detection


In [ ]:
import numpy as np
BASE_DIR = "./dataset_windows20/"

X_train = np.load(BASE_DIR + "train/X.npy")
y_train = np.load(BASE_DIR + "train/y_bin.npy")

X_val = np.load(BASE_DIR + "val/X.npy")
y_val = np.load(BASE_DIR + "val/y_bin.npy")

X_test = np.load(BASE_DIR + "test/X.npy")
y_test = np.load(BASE_DIR + "test/y_bin.npy")


In [ ]:
X_train = X_train[:, :, 1:]
X_val   = X_val[:, :, 1:]
X_test  = X_test[:, :, 1:]

print(X_train.shape)


(279986, 20, 16)


In [ ]:
X_train = X_train[:, :, 0:-1]
X_val   = X_val[:, :, 0:-1]
X_test  = X_test[:, :, 0:-1]
print(X_train.shape)

(279986, 20, 15)


In [ ]:
## Preprocessing

In [ ]:
X_train_nom = X_train[y_train == 0]   # solo nominal

mean_feat = X_train_nom.mean(axis=(0,1))   # (F,)
std_feat  = X_train_nom.std(axis=(0,1)) + 1e-8


In [ ]:
def scale_windows(X, mean, std):
    return (X - mean[None, None, :]) / std[None, None, :]

X_train_s = scale_windows(X_train, mean_feat, std_feat)
X_val_s   = scale_windows(X_val,   mean_feat, std_feat)
X_test_s  = scale_windows(X_test,  mean_feat, std_feat)


In [ ]:
import pywt
import numpy as np

def compute_dwt_windows(X, wavelet="db4", level=1):
    """
    X: (N_windows, window_size, n_channels) o (N_windows, window_size)
    return:
        A: Approximation coefficients (low-frequency)
        D: Detail coefficients (high-frequency, concatenated)
    """
    if X.ndim == 2:
        X = X[..., np.newaxis]

    N, W, F = X.shape

    A_list = []
    D_list = []

    for i in range(N):
        A_ch = []
        D_ch = []

        for ch in range(F):
            coeffs = pywt.wavedec(X[i, :, ch], wavelet, level=level)

            A = coeffs[0]                  # Approximation
            D = np.concatenate(coeffs[1:]) # All detail levels

            A_ch.append(A)
            D_ch.append(D)

        A_list.append(np.stack(A_ch, axis=1))
        D_list.append(np.stack(D_ch, axis=1))

    return np.array(A_list), np.array(D_list)
# A_windows, D_windows = compute_dwt_windows(X)
# print("A:", A_windows.shape)
# print("D:", D_windows.shape)
def adaptive_threshold_detector_HF(window, thresholds):
    win_max = np.max(window, axis=0)
    return int(np.any(win_max > thresholds))
def adaptive_threshold_detector_LF(window, thresholds):
    win_min = np.min(window, axis=0)
    return int(np.any(win_min < thresholds))
def compute_thresholds(feature_windows, y_bin, k=2.0, mode="high"):
    # feature_windows: (N, coeffs, F)
    nominal = feature_windows[y_bin == 0]

    vals = np.max(nominal, axis=1) if mode == "high" else np.min(nominal, axis=1)

    mu = np.mean(vals, axis=0)
    sigma = np.std(vals, axis=0)

    if mode == "high":
        return mu + k * sigma
    else:
        return mu - k * sigma
# TH_D = compute_thresholds(D_windows, y_bin, mode="high")  # spikes
# TH_A = compute_thresholds(A_windows, y_bin, mode="low")   # stuck
# y_pred_D = np.array([
#     adaptive_threshold_detector_HF(w, TH_D)
#     for w in D_windows
# ])

# y_pred_A = np.array([
#     adaptive_threshold_detector_LF(w, TH_A)
#     for w in A_windows
# ])



In [ ]:
A_train, D_train = compute_dwt_windows(X_train_s)
A_val,   D_val   = compute_dwt_windows(X_val_s)
A_test,  D_test  = compute_dwt_windows(X_test_s)


In [ ]:
print(D_train.shape)
# (N_samples, n_coeffs, n_channels)


(279986, 13, 15)


In [ ]:
mean = np.mean(D_train, axis=(0,1), keepdims=True)
std  = np.std(D_train, axis=(0,1), keepdims=True) + 1e-8

D_train_n = (D_train - mean) / std
D_val_n   = (D_val   - mean) / std
D_test_n  = (D_test  - mean) / std


In [ ]:
import tensorflow as tf
model = tf.keras.Sequential([

    tf.keras.layers.Conv1D(
        filters=8,
        kernel_size=3,
        padding="same",
        activation="relu",
        input_shape=(D_train_n.shape[1], D_train_n.shape[2])
    ),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Conv1D(
        filters=8,
        kernel_size=3,
        padding="same",
        activation="relu"
    ),

    tf.keras.layers.GlobalAveragePooling1D(),

    tf.keras.layers.Dense(8, activation="relu"),

    tf.keras.layers.Dense(1, activation="sigmoid")
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
history = model.fit(
    D_train_n,
    y_train,
    validation_data=(D_val_n, y_val),
    epochs=30,
    batch_size=64,
    verbose=1
)


Epoch 1/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 24s 5ms/step - accuracy: 0.8480 - loss: 0.3638 - val_accuracy: 0.9301 - val_loss: 0.2299
Epoch 2/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 23s 5ms/step - accuracy: 0.9120 - loss: 0.2530 - val_accuracy: 0.9275 - val_loss: 0.2095
Epoch 3/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - accuracy: 0.9169 - loss: 0.2388 - val_accuracy: 0.9300 - val_loss: 0.2071
Epoch 4/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - accuracy: 0.9189 - loss: 0.2332 - val_accuracy: 0.9330 - val_loss: 0.2006
Epoch 5/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - accuracy: 0.9192 - loss: 0.2313 - val_accuracy: 0.9353 - val_loss: 0.1950
Epoch 6/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - accuracy: 0.9217 - loss: 0.2269 - val_accuracy: 0.9366 - val_loss: 0.1909
Epoch 7/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 19s 4ms/step - accuracy: 0.9217 - loss: 0.2261 - val_accuracy: 0.9417 - val_loss: 0.1885
Epoch 8/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - accuracy: 0.9241 - loss: 0

In [ ]:
y_pred = (model.predict(D_test_n) > 0.5).astype(int)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, digits=4))


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
              precision    recall  f1-score   support

           0     0.9128    0.9923    0.9509     30406
           1     0.9913    0.9026    0.9449     29572

    accuracy                         0.9481     59978
   macro avg     0.9521    0.9475    0.9479     59978
weighted avg     0.9515    0.9481    0.9479     59978



In [ ]:
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 13, 8)          │           368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 13, 8)          │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 13, 8)          │           200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 8)              │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,013 (7.87 KB)

 Trainable params: 665 (2.60 KB)

 Non-trainable params: 16 (64.00 B)

 Optimizer params: 1,332 (5.21 KB)

Se implementó un modelo de detección de anomalías basado en una red neuronal convolucional 1D utilizando directamente los coeficientes de detalle (D) obtenidos mediante la Transformada Wavelet Discreta (DWT) con wavelet “db4” y nivel 1. Primero, cada ventana temporal multicanal fue transformada al dominio wavelet mediante pywt.wavedec, extrayendo y concatenando únicamente los coeficientes de alta frecuencia (D), ya que estos contienen la información asociada a cambios rápidos y posibles anomalías. Posteriormente, los coeficientes fueron normalizados utilizando la media y desviación estándar calculadas sobre el conjunto de entrenamiento para garantizar estabilidad numérica y mejorar la convergencia. Estos tensores normalizados, con dimensión (n_coeficientes, n_canales), fueron utilizados como entrada de una CNN 1D ligera compuesta por dos capas convolucionales con activación ReLU, seguidas de Batch Normalization, una capa Global Average Pooling para reducción de dimensionalidad y dos capas densas finales para clasificación binaria mediante función sigmoide. El modelo fue entrenado con función de pérdida binary cross-entropy y optimizador Adam, evaluando su desempeño en términos de accuracy, recall y matriz de confusión sobre el conjunto de prueba.